# Smoke test — RT-DETRv2 SportsMOT Kaggle environment

Validates in a **few minutes** that the Kaggle session can run the real
`train.kaggle.ipynb` (which takes 3–5 hours), before spending GPU quota on it:

1. **System requirements** — GPU present, torch has CUDA kernels for its compute
   capability (the check that catches the P100 "no kernel image is available"
   failure), fp16 works, disk and internet available.
2. **Dependencies** — the exact pinned pip install the trainer uses.
3. **Hub auth** — whether `HF_TOKEN` is readable from the `external-secrets`
   dataset (report-only; nothing is pushed).
4. **Data mount + mock training** — verifies the pre-staged `sportsmot-train`
   Kaggle Dataset is mounted (what makes the real trainer start instantly), then
   runs the trainer's real data pipeline, augmentation, model and loss on a tiny
   slice: one sequence's first ~99 frames, 25 optimizer steps. Falls back to a
   small Hub download (~25 MB) if the mount is absent.
5. **Inference + serialization** — post-processing and a save/reload round-trip.

Launched headless the same way as the trainer (`enable_gpu` +
`machine_shape: NvidiaTeslaT4` + the mounts from the adjacent
`smoketest.config.json`). Any CRITICAL failure aborts immediately with a clear message in the log.


In [ ]:
# --- 1. System requirements ------------------------------------------------
# Fail fast, loudly, and specifically: every check prints PASS/FAIL/WARN, and
# any CRITICAL failure aborts the run before GPU minutes are wasted.
import os

# Same single-GPU pin as train.kaggle.ipynb: with two visible GPUs, Trainer
# wraps the model in torch.nn.DataParallel, whose input scatter splits every
# tensor along dim 0 — including each image's variable-length [num_boxes, 4]
# label tensors — corrupting the detection targets. Set before CUDA init.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import shutil, socket, sys
import torch

failures = []

def check(name, ok, detail="", critical=True):
    tag = "PASS" if ok else ("FAIL" if critical else "WARN")
    print(f"[{tag}] {name}: {detail}")
    if not ok and critical:
        failures.append(name)

check("python >= 3.10", sys.version_info >= (3, 10), sys.version.split()[0])
check("torch", True, f"{torch.__version__} (built for CUDA {torch.version.cuda})")

cuda_ok = torch.cuda.is_available()
check("cuda available", cuda_ok,
      f"{torch.cuda.device_count()} visible device(s)" if cuda_ok
      else "no GPU — select the 'GPU T4 x2' accelerator")

if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    check("gpu", True, f"{props.name}, {props.total_memory / 1e9:.1f} GB")
    check("gpu memory >= 10 GB", props.total_memory >= 10e9,
          "needed for batch-4 640px training")

    # THE check that caught the P100 failure: torch must ship compiled kernels
    # for this GPU's compute capability, or every CUDA op raises
    # "no kernel image is available for execution on the device".
    cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
    archs = torch.cuda.get_arch_list()
    check("torch kernels for this GPU", cap in archs,
          f"GPU is {cap}; torch built for {archs}")

    # Prove it end to end with a real kernel launch in fp16 (what training uses).
    try:
        x = torch.randn(64, 64, device="cuda", dtype=torch.float16)
        check("fp16 CUDA op", bool(torch.isfinite((x @ x).sum()).item()),
              "matmul executed on device")
    except Exception as e:
        check("fp16 CUDA op", False, f"{type(e).__name__}: {e}")

    # Report NATIVE bf16 only (Ampere+, sm_80). torch.cuda.is_bf16_supported()
    # is unusable for this: since torch ~2.6 it returns True on Turing because
    # bf16 can be EMULATED in software — several times slower than fp16.
    check("native bf16", torch.cuda.get_device_capability(0) >= (8, 0),
          "T4 (Turing) has none — trainer must use fp16", critical=False)

disk_free = shutil.disk_usage("/kaggle/working").free / 1e9
check("disk space /kaggle/working >= 15 GB", disk_free >= 15,
      f"{disk_free:.0f} GB free (real trainer's checkpoints need ~15 GB)")

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
    check("internet -> huggingface.co", True, "reachable")
except OSError as e:
    check("internet -> huggingface.co", False,
          f"{e} — enable Internet in session options")

if failures:
    raise RuntimeError(f"CRITICAL system checks failed: {failures} — "
                       "fix these before running the real trainer.")
print("\nAll critical system checks passed.")


In [ ]:
# --- 2. Dependencies (identical to train.kaggle.ipynb) -------------------
# The exact pinned install the real trainer performs, so this smoke test
# validates it: only torch-independent extras, with torch constrained to
# Kaggle's preinstalled (working) build so nothing can swap it out.
import importlib.metadata
import subprocess, tempfile

_con = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
_con.write(f"torch=={torch.__version__}\n")
_con.close()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", _con.name,
     "albumentations>=1.4.10", "pycocotools", "torchmetrics"],
    check=True,
)

assert importlib.metadata.version("torch") == torch.__version__, \
    "pip swapped torch out despite the constraint — do not proceed"

import albumentations, torchmetrics, pycocotools  # noqa: F401  (import check)
import accelerate, huggingface_hub, transformers
from transformers.models import rt_detr_v2  # noqa: F401  RT-DETRv2 code present
print(f"transformers={transformers.__version__} accelerate={accelerate.__version__} "
      f"huggingface_hub={huggingface_hub.__version__} "
      f"albumentations={albumentations.__version__} torchmetrics={torchmetrics.__version__}")
print("Dependency install OK; torch untouched; RT-DETRv2 model code available.")


In [ ]:
# --- 3. Hub authentication (report-only) -----------------------------------
# The smoke test never pushes anything to the Hub — it only reports whether the
# real trainer WOULD be able to. Missing token is a warning, not a failure.
# Run 1 found nothing at the assumed /kaggle/input/external-secrets/secrets even
# with the dataset referenced in the kernel metadata, so instead of hard-coding
# a path we print everything mounted under /kaggle/input (the diagnostic that
# matters) and scan for a file named "secrets" wherever it is.
from pathlib import Path

from huggingface_hub import HfApi


def _load_env_file(path):
    env = {}
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            env[key.strip()] = val.strip().strip('"').strip("'")
    return env


def _find_secrets_file():
    base = Path("/kaggle/input")
    mounts = sorted(p.name for p in base.iterdir()) if base.is_dir() else []
    print(f"/kaggle/input mounts: {mounts or 'NONE'}")
    for m in mounts:
        files = sorted(str(p.relative_to(base)) for p in (base / m).rglob("*") if p.is_file())
        print(f"  {m}/: {files[:20]}{' ...' if len(files) > 20 else ''}")
    hits = sorted(base.rglob("secrets")) if mounts else []
    return hits[0] if hits else None


_secrets = _find_secrets_file()
try:
    os.environ["HF_TOKEN"] = _load_env_file(_secrets)["HF_TOKEN"]
    print(f"HF_TOKEN found at {_secrets} — authenticated on the Hub as:",
          HfApi().whoami()["name"])
except Exception as e:
    print(f"[WARN] no usable HF_TOKEN ({type(e).__name__}: {e}) — the real "
          "trainer would save to /kaggle/working instead of pushing to the Hub.")


In [ ]:
# --- 4. Tiny dataset slice --------------------------------------------------
# Preferred source — same as the real trainer: the pre-staged sportsmot-train
# Kaggle Dataset mounted under /kaggle/input (created by
# the data-preparation staging kernel (prepare-data.kaggle.ipynb)). Validating the mount here is the
# point: a missing mount means the real trainer would fall back to a ~28k-file
# Hub download that previously ate an entire GPU session.
# Fallback: pull ONE sequence's gt.txt + first <=99 frames from the Hub (~25 MB).
import random
from pathlib import Path

from huggingface_hub import snapshot_download

DATASET_ID = "Lekim89/sportsmot"
CHECKPOINT = "PekingU/rtdetr_v2_r50vd"
SMOKE_SEQ = "v_-6Os86HzwCs_c009"   # the basketball sequence the trainer holds out
IMAGE_SIZE = 640
SEED = 42
N_TRAIN, N_VAL = 40, 8

random.seed(SEED)
torch.manual_seed(SEED)

# Same precision auto-detect as the trainer (T4 -> fp16). Gated on compute
# capability, NOT is_bf16_supported(), which counts slow emulated bf16 on Turing.
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0) >= (8, 0)
USE_FP16 = torch.cuda.is_available() and not USE_BF16


def _find_mounted_dataset():
    """Same probe as the trainer: extracted tree first, then the staged tar."""
    import subprocess
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for pattern in ("*/train", "*/*/train", "*/*/*/train"):
        for cand in sorted(base.glob(pattern)):
            if any(cand.glob("*/gt/gt.txt")):
                return cand.parent
    for pattern in ("*/sportsmot-train.tar", "*/*/sportsmot-train.tar",
                    "*/*/*/sportsmot-train.tar"):
        for tar_path in sorted(base.glob(pattern)):
            dest = Path("/kaggle/working/sportsmot-data")
            if not (dest / "train").is_dir():
                print(f"extracting {tar_path} (validates the trainer's untar path)...")
                dest.mkdir(parents=True, exist_ok=True)
                subprocess.run(["tar", "-xf", str(tar_path), "-C", str(dest)], check=True)
            return dest
    return None


root = _find_mounted_dataset()
if root is not None:
    n_seq = sum(1 for _ in (root / "train").glob("*/gt/gt.txt"))
    print(f"[PASS] sportsmot data staged: {root} ({n_seq} annotated sequences) — "
          "the real trainer will start without a Hub download")
    assert (root / "train" / SMOKE_SEQ / "gt" / "gt.txt").is_file(), \
        f"staged data is missing {SMOKE_SEQ} — re-run the data-preparation staging kernel"
else:
    print("[WARN] no staged SportsMOT data mounted — the real trainer would "
          "fall back to the SLOW ~28k-file Hub download. Run the "
          "data-preparation/ staging kernel. Smoke test continues with a tiny Hub slice.")
    root = Path(snapshot_download(
        DATASET_ID, repo_type="dataset",
        allow_patterns=[f"train/{SMOKE_SEQ}/gt/*", f"train/{SMOKE_SEQ}/img1/0000*.jpg"],
    ))

imgs = sorted((root / "train" / SMOKE_SEQ / "img1").glob("*.jpg"))[:99]
print(f"using {len(imgs)} frames of {SMOKE_SEQ}")
assert len(imgs) >= N_TRAIN + N_VAL, \
    f"expected >= {N_TRAIN + N_VAL} frames, got {len(imgs)} — dataset layout changed?"


In [ ]:
# --- 5. Annotations -> samples (same parsing as the trainer) ---------------
from collections import defaultdict

def parse_gt(gt_path: Path) -> dict:
    """Return {frame_number: [[x, y, w, h], ...]} from a MOTChallenge gt.txt."""
    frames = defaultdict(list)
    for line in gt_path.read_text().strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        frame = int(parts[0])
        x, y, w, h = (float(v) for v in parts[2:6])
        conf = int(parts[6])
        if conf == 0 or w <= 0 or h <= 0:
            continue
        frames[frame].append([x, y, w, h])
    return frames

gt = parse_gt(root / "train" / SMOKE_SEQ / "gt" / "gt.txt")
samples = [(p, gt[int(p.stem)]) for p in imgs if gt.get(int(p.stem))]
assert len(samples) >= N_TRAIN + N_VAL, \
    f"only {len(samples)} annotated frames — gt.txt parsing or coverage problem"
train_samples = samples[:N_TRAIN]
val_samples = samples[N_TRAIN:N_TRAIN + N_VAL]
print(f"{len(train_samples)} train / {len(val_samples)} val frames, "
      f"~{sum(len(b) for _, b in train_samples) / len(train_samples):.1f} boxes/frame")


In [ ]:
# --- 6. Dataset + augmentation (same as the trainer) -----------------------
import albumentations as A
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

train_aug = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
    ],
    bbox_params=A.BboxParams(format="coco", label_fields=["labels"], clip=True, min_area=4),
)


class MotDataset(Dataset):
    """Yields the pixel_values + COCO-style labels that RT-DETRv2 expects."""

    def __init__(self, samples, processor, augment):
        self.samples = samples
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, boxes = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        labels = [0] * len(boxes)          # single class: 0 == "player"
        if self.augment:
            out = train_aug(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = out["image"], out["bboxes"], out["labels"]
        annotations = {
            "image_id": idx,
            "annotations": [
                {"bbox": list(b), "category_id": c, "area": b[2] * b[3], "iscrowd": 0}
                for b, c in zip(boxes, labels)
            ],
        }
        encoding = self.processor(images=image, annotations=annotations, return_tensors="pt")
        return {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": encoding["labels"][0],
        }


def collate_fn(batch):
    # Detection targets have variable length, so labels stay a list (not stacked).
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": [b["labels"] for b in batch],
    }


In [ ]:
# --- 7. Model + processor (same as the trainer) ----------------------------
from transformers import AutoImageProcessor, AutoModelForObjectDetection

processor = AutoImageProcessor.from_pretrained(
    CHECKPOINT,
    do_resize=True,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    use_fast=True,
)
model = AutoModelForObjectDetection.from_pretrained(
    CHECKPOINT,
    id2label={0: "player"},
    label2id={"player": 0},
    anchor_image_size=None,
    ignore_mismatched_sizes=True,
)

train_ds = MotDataset(train_samples, processor, augment=True)
val_ds = MotDataset(val_samples, processor, augment=False)
print("model + processor loaded")


In [ ]:
# --- 8. Mock training: 25 optimizer steps ----------------------------------
# Same optimizer/precision settings as the real trainer, shrunk to finish in a
# couple of minutes. A finite, decreasing-ish loss proves forward, loss (incl.
# scipy Hungarian matching), backward and optimizer all work on this GPU.
import math
import time

from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="smoketest",
    max_steps=25,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    weight_decay=1e-4,
    max_grad_norm=0.1,
    warmup_steps=5,
    bf16=USE_BF16,
    fp16=USE_FP16,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=5,
    logging_first_step=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,        # the model needs our custom "labels" field
    report_to="none",
    seed=SEED,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

t0 = time.time()
result = trainer.train()
elapsed = time.time() - t0
assert math.isfinite(result.training_loss), f"training loss is {result.training_loss}"
print(f"25 steps in {elapsed:.0f}s ({elapsed / 25:.1f}s/step) — "
      f"mean training loss {result.training_loss:.3f}")

eval_metrics = trainer.evaluate()
assert math.isfinite(eval_metrics["eval_loss"]), f"eval loss is {eval_metrics['eval_loss']}"
print(f"eval loss on {len(val_ds)} held-out frames: {eval_metrics['eval_loss']:.3f}")


In [ ]:
# --- 9. Inference + save/reload round-trip ---------------------------------
# Post-process one val frame (what inference.py does) and verify the model
# serializes and loads back — the two things the trainer does after training.
model.eval()
img_path, boxes = val_samples[0]
image = Image.open(img_path).convert("RGB")
inputs = processor(images=image, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
(det,) = processor.post_process_object_detection(
    outputs,
    target_sizes=torch.tensor([image.size[::-1]]).to(model.device),
    threshold=0.3,
)
print(f"detected {len(det['boxes'])} boxes on one val frame "
      f"(ground truth has {len(boxes)}; counts need not match after 25 steps)")

OUT_DIR = "/kaggle/working/smoketest-model"
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
AutoModelForObjectDetection.from_pretrained(OUT_DIR)
AutoImageProcessor.from_pretrained(OUT_DIR)
print("save/reload round-trip OK ->", OUT_DIR)


In [ ]:
# --- 10. Verdict ------------------------------------------------------------
print("=" * 62)
print("SMOKE TEST PASSED — this environment can run train.kaggle.ipynb")
print("=" * 62)
